<a href="https://colab.research.google.com/github/AnshuRajbhar/Anshu.demo/blob/main/BDM_GA3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%load_ext cudf.pandas

/usr/local/lib/python3.11/dist-packages/cudf/utils/gpu_utils.py:75: UserWarning: Failed to dlopen libcuda.so.1
  warnings.warn(str(e))
/usr/local/lib/python3.11/dist-packages/cudf/pandas/__init__.py:64: UserWarning: Function "cuInit" not found
  warnings.warn(str(e))


In [ ]:
import pandas as pd
from google.colab import files

print("Upload file in .xlsx format which you will download from dataset given for you")
file = "dataset_3_90.xlsx"

uploaded = files.upload()
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
file = next(iter(uploaded))


sheet_names = ["Data", "Cost", "Shift_Running", "Actual_Output", "Scrap"]
data_df = pd.read_excel(file, sheet_name=sheet_names[0])

data_df = data_df.dropna(axis=1, how="all")
data_df["Revenue"] = data_df["Sales Quantity"] * data_df["Price"]
cost_df = pd.read_excel(file, sheet_name=sheet_names[1])
cost_df = cost_df.dropna(axis=1, how="all")
cost_df["Unit_Cost"] = cost_df["Direct Materials"] + cost_df["Direct Labour"] + cost_df["Production Overhead"] + cost_df["G&A Overhead"] + cost_df["Finance Costs"]
shift_running_df = pd.read_excel(file, sheet_name=sheet_names[2])
shift_running_df = shift_running_df.dropna(axis=1, how="all")
actual_output_df = pd.read_excel(file, sheet_name=sheet_names[3])
actual_output_df = actual_output_df.dropna(axis=1, how="all")
scrap_df = pd.read_excel(file, sheet_name=sheet_names[4])
scrap_df = scrap_df.dropna(axis=1, how="all")

Upload file in .xlsx format which you will download from dataset given for you


Saving dataset_3_353.xlsx to dataset_3_353 (4).xlsx
User uploaded file "dataset_3_353 (4).xlsx" with length 15909 bytes


<a href="https://docs.google.com/spreadsheets/d/1SQxaF5bc1j1IcU8xs02P-1380QreDjHT/edit?usp=sharing&ouid=111673709741205601338&rtpof=true&sd=true" >Google sheet link with solution </a>

# Question 1
Which BS4 only Gear Assembly saw the maximum sales in the first quarter of any given year?

Go to the 'Data' sheet.
Add a new column 'Total' with 	formula:
`=G2 * H2 (Sales Quantity * Price).`
Create a Pivot Table:
- Rows: Gear Assembly
- Filters: GA Category = 'BS4 Only', Quarter = 'Q1'
- Values: SUM of 'Total'
Find the assembly with the 	highest total sales.

i created a demo pivot table in sheet "q1_gear_assembly_salewise_pivot"

In [14]:
data_df["Revenue"] = data_df["Sales Quantity"] * data_df["Price"]
bs4_only = data_df[data_df["GA Category"] == "BS4 Only"]
q1_bs4 = bs4_only[bs4_only["Quarter"] == "Q1"]
q1_top_sales_per_year = (
    q1_bs4.groupby(["Fiscal Year", "Gear Assembly"])["Revenue"]
    .sum()
    .reset_index()
    .sort_values("Revenue", ascending=False)
    .groupby("Fiscal Year")
    .first()
)
print(q1_top_sales_per_year)


                     Gear Assembly  Revenue
Fiscal Year                                
2019-20      Gear Assembly 1 (BS4)  8472456
2020-21      Gear Assembly 2 (BS4)  7566568
2021-22      Gear Assembly 1 (BS4)  7580888


# Question 2
Based on the data, which is Gear Assembly is incurring maximum (net) loss?

- Create another helper column 'Unit Cost' in Cost sheet (assuming Ith column) which will be sum of all costs
```
= D2 + E2 + F2 + G2 + H2
```
- Lookup the Unit cost from Cost sheet to Data sheet (at Jth column)
 ```
 =VLOOKUP(A2&E2,Cost!$A$1:$I$19,9)
 ```
- Create another helper column 'Unit Margin' in Data sheet (Kth column)

- Just find the Gear assembly with minimum Unit Margin

In [ ]:
merged_df = data_df.merge(cost_df, left_on=["Gear Assembly", "Fiscal Year"], right_on=["SALES DETAILS (GEAR ASSEMBLIES)", "FY"], how="outer")
merged_df["Unit_Cost"] = merged_df["Direct Materials"] + merged_df["Direct Labour"] + merged_df["Production Overhead"] + merged_df["G&A Overhead"] + merged_df["Finance Costs"]
merged_df["Unit_Margin"] = merged_df["Price"] - merged_df["Unit_Cost"]
merged_df = merged_df.reset_index(drop=True)
minidx = merged_df["Unit_Margin"].idxmin()
answer_2 = merged_df.loc[minidx, "Gear Assembly"]
print("Question 2:", answer_2)
merged_df.sort_values("Unit_Margin", ascending=True).head(3)

Question 2: Gear Assembly 3 (BS4/6)


,Gear Assembly,GA Category,Month,Quarter,Fiscal Year,Quantity Produced,Sales Quantity,Price,Revenue,SALES DETAILS (GEAR ASSEMBLIES),FY,Direct Materials,Direct Labour,Production Overhead,G&A Overhead,Finance Costs,Unit_Cost,Unit_Margin
82,Gear Assembly 3 (BS4/6),Combination,April,Q1,2021-22,6363,5568,354,1971072,Gear Assembly 3 (BS4/6),2021-22,245,80,150,55,34,564,-210
76,Gear Assembly 3 (BS4/6),Combination,October,Q3,2020-21,6827,6048,341,2062368,Gear Assembly 3 (BS4/6),2020-21,242,75,121,46,32,516,-175
72,Gear Assembly 3 (BS4/6),Combination,June,Q1,2020-21,6665,5725,341,1952225,Gear Assembly 3 (BS4/6),2020-21,242,75,121,46,32,516,-175


# Question 3
Which Gear Assembly returned the highest percentage unit (net) margin?

Calculate Unit margin percentage (Mth column) (assuming you kept all helper columns which we created in prev question)
```
=K2/H2*100
```
Take pivot table of Data sheet refer same sheet `q2q3_gear_assembly_margin_pivot` <br/>
Add the unit margin percentage to the values. I am considering the average of all margin percentage values for a particular assembly. You may consider the median or another MST because of the large gap between margins; it does not matter, it giving the same answer in all cases for me.

In [ ]:
merged_df["Unit_Margin_Percentage"] = (merged_df["Unit_Margin"] / merged_df["Price"]) * 100
avg_margin_pct = merged_df.groupby("Gear Assembly")["Unit_Margin_Percentage"].mean()
answer_3 = avg_margin_pct.idxmax()
print("Question 3:", answer_3)
avg_margin_pct.sort_values(ascending=False).head(3)

Question 3: Gear Assembly 5 (BS6)


,Unit_Margin_Percentage
Gear Assembly,
Gear Assembly 5 (BS6),20.990154
Gear Assembly 2 (BS4),16.139597
Gear Assembly 4 (BS4/6),9.625990


# Question 4
Which period saw the least amount of ending inventory in terms of volume?

- calculate new columns (Nth,Oth) Period and Ending Inventory
```
=CONCAT(D2,E2)
```
```
=F2-G2
```
Take Pivot sheet refer `q4period_inventory_pivot`
Period in rows, Ending Inventory as values(sum)

  (in the question they did not mentioned what should we return as answer but according to the hint i think we should return the period =conat(qrtr,fy) let me know if you think it should be something else)

In [ ]:
data_df["Period"] = data_df["Quarter"] + data_df["Fiscal Year"]
data_df["Ending_Inventory"] = data_df["Quantity Produced"] - data_df["Sales Quantity"]
period_inventory = data_df.groupby("Period")["Ending_Inventory"].sum()
answer_4 = period_inventory.idxmin()
print("Question 4:", answer_4)
period_inventory.sort_values().head(3)

Question 4: Q22021-22


,Ending_Inventory
Period,
Q22021-22,6299
Q32020-21,6916
Q12021-22,7695


# Question 5
Which Gear Assembly made the maximum jump in the percentage revenue from 2019-20 to 2020-21?

 - Take Pivot table
  Rows: Gear Assembly
  Columns: Fiscal Year
  Values: Revenue (Summarize by SUM)
 - calculate Revenue jump percentage and get the maximum value
```
=(C3-B3)/B3*100
```
Refer `q5gear_assembly_revenue_pivot`

In [ ]:
revenue_data = data_df.groupby(["Gear Assembly", "Fiscal Year"])["Revenue"].sum().reset_index()
revenue_2019 = revenue_data[revenue_data["Fiscal Year"] == "2019-20"].set_index("Gear Assembly")["Revenue"]
revenue_2020 = revenue_data[revenue_data["Fiscal Year"] == "2020-21"].set_index("Gear Assembly")["Revenue"]
revenue_change = ((revenue_2020 - revenue_2019) / revenue_2019 * 100).fillna(0)
answer_5 = revenue_change.idxmax()
print("Question 5:", answer_5)
revenue_change.sort_values(ascending=False).head(3)

Question 5: Gear Assembly 5 (BS6)


,Revenue
Gear Assembly,
Gear Assembly 5 (BS6),22.594678
Gear Assembly 6 (BS6),6.752164
Gear Assembly 2 (BS4),4.040130


# Question 6
What is the Overall Equipment Effectiveness (OEE) of manufacturing in Week-1 (01-04-2022 to 07-04-2022 both days included)? (FLOAT BETWEEN 0 AND 1)

OEE of Week-1 (01–07 April 2022)
  - Step 1: Availability In Shift_Running sheet: E is helper cell `=ARRAYFORMULA(IF(A2:A="", "", TO_DATE(A2:A)))`
  ```
=SUMPRODUCT((A2:A >= DATE(2022,4,1)) * (A2:A <= DATE(2022,4,7)) * ((B2:D) = "Operational"))
  ```
  - Step 2: Performance In Actual_Output sheet:
  

In [ ]:
shift_running_df["Date"] = pd.to_datetime(shift_running_df["Date"])
actual_output_df["Date"] = pd.to_datetime(actual_output_df["Date"])
scrap_df["Date"] = pd.to_datetime(scrap_df["Date"])

week1_start = pd.to_datetime("2022-04-01")
week1_end = pd.to_datetime("2022-04-07")

week1_shifts = shift_running_df[(shift_running_df["Date"] >= week1_start) & (shift_running_df["Date"] <= week1_end)]
week1_output = actual_output_df[(actual_output_df["Date"] >= week1_start) & (actual_output_df["Date"] <= week1_end)]
week1_scrap = scrap_df[(scrap_df["Date"] >= week1_start) & (scrap_df["Date"] <= week1_end)]

shift_running_cols = ["Shift 1 (8 Hours)", "Shift 2 (8 Hours)", "Shift 3 (8 Hours)"]
output_cols = ["Shift 1", "Shift 2", "Shift 3"]

availability_week1 = (week1_shifts[shift_running_cols] == "Operational").sum().sum() / (week1_shifts.shape[0] * 3)
actual_production_week1 = week1_output[output_cols].sum().sum()
good_production_week1 = actual_production_week1 - week1_scrap[output_cols].sum().sum()
operational_shifts_week1 = (week1_shifts[shift_running_cols] == "Operational").sum().sum()
performance_week1 = actual_production_week1 / (operational_shifts_week1 * 4000) if operational_shifts_week1 > 0 else 0
quality_week1 = good_production_week1 / actual_production_week1 if actual_production_week1 > 0 else 0
oee_week1 = availability_week1 * performance_week1 * quality_week1
answer_6 = oee_week1
print("Question 6:", answer_6)

Question 6: 0.8872380952380953


# Question 7
What is the overall quality of the manufacturing process during the fortnight? (FLOAT BETWEEN 0 AND 1)

In [ ]:
total_actual_output = actual_output_df[output_cols].sum().sum()
total_scrap = scrap_df[output_cols].sum().sum()
total_good_output = total_actual_output - total_scrap
quality_fortnight = total_good_output / total_actual_output if total_actual_output > 0 else 0
answer_7 = quality_fortnight
print("Question 7:", answer_7)

Question 7: 0.9946875436879631


# Question 8
What is the performance of the manufacturing process during Week-2? (FLOAT BETWEEN 0 AND 1)

In [ ]:
week2_start = pd.to_datetime("2022-04-08")
week2_end = pd.to_datetime("2022-04-14")

week2_output = actual_output_df[(actual_output_df["Date"] >= week2_start) & (actual_output_df["Date"] <= week2_end)]
week2_scrap = scrap_df[(scrap_df["Date"] >= week2_start) & (scrap_df["Date"] <= week2_end)]

total_production_week2 = week2_output[output_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum().sum()
total_scrap_week2 = week2_scrap[output_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum().sum()

if total_production_week2 > 0:
    performance_week2 = (total_production_week2 - total_scrap_week2) / total_production_week2
else:
    performance_week2 = 0

answer_8 = performance_week2
print("Question 8:", answer_8)


Question 8: 0.9944096244754873


# Question 9
What is the average number of parts manufactured per hour during the fortnight? (INTEGER)

In [ ]:
operational_shifts_total = (shift_running_df[shift_running_cols] == "Operational").sum().sum()
total_production_hours = operational_shifts_total * 8
parts_per_hour = total_actual_output / total_production_hours if total_production_hours > 0 else 0
answer_9 = int(parts_per_hour)
print("Question 9:", answer_9)

# Question 10
The company uses MAPE (Mean Absolute Percentage Error) to measure process variability in the manufacturing process. Which shift sees the maximum process variability during the fortnight? (STRING)

In [ ]:
rated_output = 4000
shift_data = []
for i, (shift_running_col, output_col) in enumerate(zip(shift_running_cols, output_cols)):
    operational_mask = shift_running_df[shift_running_col] == "Operational"
    if operational_mask.sum() > 0:
        actual_outputs = actual_output_df.loc[operational_mask, output_col]
        summ = (abs(actual_outputs - rated_output) / rated_output).sum()
        mape = summ/actual_outputs.shape[0]
        shift_data.append((f"Shift {i+1}", mape))

max_mape_shift = max(shift_data, key=lambda x: x[1])[0]
answer_10 = max_mape_shift
print("Question 10:", answer_10)

#All Questions

In [ ]:
# 1. Which BS4 only Gear Assembly saw the maximum sales in the first quarter of any given year?
bs4_only = data_df[data_df["GA Category"] == "BS4 Only"]
q1_bs4 = bs4_only[bs4_only["Quarter"] == "Q1"]
q1_sales = q1_bs4.groupby("Gear Assembly")["Revenue"].sum()
answer_1 = q1_sales.idxmax()
print("Question 1:", answer_1)

# 2. Based on the data, which is Gear Assembly is incurring maximum (net) loss?
merged_df = data_df.merge(cost_df, left_on=["Gear Assembly", "Fiscal Year"], right_on=["SALES DETAILS (GEAR ASSEMBLIES)", "FY"], how="outer")
merged_df["Unit_Cost"] = merged_df["Direct Materials"] + merged_df["Direct Labour"] + merged_df["Production Overhead"] + merged_df["G&A Overhead"] + merged_df["Finance Costs"]
merged_df["Unit_Margin"] = merged_df["Price"] - merged_df["Unit_Cost"]
merged_df = merged_df.reset_index(drop=True)
minidx = merged_df["Unit_Margin"].idxmin()
answer_2 = merged_df.loc[minidx, "Gear Assembly"]
print("Question 2:", answer_2)

# 3. Which Gear Assembly returned the highest percentage unit (net) margin?
merged_df["Unit_Margin_Percentage"] = (merged_df["Unit_Margin"] / merged_df["Price"]) * 100
avg_margin_pct = merged_df.groupby("Gear Assembly")["Unit_Margin_Percentage"].mean()
answer_3 = avg_margin_pct.idxmax()
print("Question 3:", answer_3)

# 4. Which period saw the least amount of ending inventory in terms of volume?
data_df["Period"] = data_df["Quarter"] + data_df["Fiscal Year"]
data_df["Ending_Inventory"] = data_df["Quantity Produced"] - data_df["Sales Quantity"]
period_inventory = data_df.groupby("Period")["Ending_Inventory"].sum()
answer_4 = period_inventory.idxmin()
print("Question 4:", answer_4)

# 5. Which Gear Assembly made the maximum jump in the percentage revenue from 2019-20 to 2020-21?
revenue_data = data_df.groupby(["Gear Assembly", "Fiscal Year"])["Revenue"].sum().reset_index()
revenue_2019 = revenue_data[revenue_data["Fiscal Year"] == "2019-20"].set_index("Gear Assembly")["Revenue"]
revenue_2020 = revenue_data[revenue_data["Fiscal Year"] == "2020-21"].set_index("Gear Assembly")["Revenue"]
revenue_change = ((revenue_2020 - revenue_2019) / revenue_2019 * 100).fillna(0)
answer_5 = revenue_change.idxmax()
print("Question 5:", answer_5)

# 6. What is the Overall Equipment Effectiveness (OEE) of manufacturing in Week-1 (01-04-2022 to 07-04-2022 both days included)?
shift_running_df["Date"] = pd.to_datetime(shift_running_df["Date"])
actual_output_df["Date"] = pd.to_datetime(actual_output_df["Date"])
scrap_df["Date"] = pd.to_datetime(scrap_df["Date"])

week1_start = pd.to_datetime("2022-04-01")
week1_end = pd.to_datetime("2022-04-07")

week1_shifts = shift_running_df[(shift_running_df["Date"] >= week1_start) & (shift_running_df["Date"] <= week1_end)]
week1_output = actual_output_df[(actual_output_df["Date"] >= week1_start) & (actual_output_df["Date"] <= week1_end)]
week1_scrap = scrap_df[(scrap_df["Date"] >= week1_start) & (scrap_df["Date"] <= week1_end)]

shift_running_cols = ["Shift 1 (8 Hours)", "Shift 2 (8 Hours)", "Shift 3 (8 Hours)"]
output_cols = ["Shift 1", "Shift 2", "Shift 3"]

availability_week1 = (week1_shifts[shift_running_cols] == "Operational").sum().sum() / (week1_shifts.shape[0] * 3)
actual_production_week1 = week1_output[output_cols].sum().sum()
good_production_week1 = actual_production_week1 - week1_scrap[output_cols].sum().sum()
operational_shifts_week1 = (week1_shifts[shift_running_cols] == "Operational").sum().sum()
performance_week1 = actual_production_week1 / (operational_shifts_week1 * 4000) if operational_shifts_week1 > 0 else 0
quality_week1 = good_production_week1 / actual_production_week1 if actual_production_week1 > 0 else 0
oee_week1 = availability_week1 * performance_week1 * quality_week1
answer_6 = oee_week1
print("Question 6:", answer_6)

# 7. What is the overall quality of the manufacturing process during the fortnight?
total_actual_output = actual_output_df[output_cols].sum().sum()
total_scrap = scrap_df[output_cols].sum().sum()
total_good_output = total_actual_output - total_scrap
quality_fortnight = total_good_output / total_actual_output if total_actual_output > 0 else 0
answer_7 = quality_fortnight
print("Question 7:", answer_7)

# 8. What is the performance of the manufacturing process during Week-2?
week2_start = pd.to_datetime("2022-04-08")
week2_end = pd.to_datetime("2022-04-14")

week2_output = actual_output_df[(actual_output_df["Date"] >= week2_start) & (actual_output_df["Date"] <= week2_end)]
week2_scrap = scrap_df[(scrap_df["Date"] >= week2_start) & (scrap_df["Date"] <= week2_end)]

total_production_week2 = week2_output[output_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum().sum()
total_scrap_week2 = week2_scrap[output_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum().sum()

if total_production_week2 > 0:
    performance_week2 = (total_production_week2 - total_scrap_week2) / total_production_week2
else:
    performance_week2 = 0

answer_8 = performance_week2
print("Question 8:", answer_8)

# 9. What is the average number of parts manufactured per hour during the fortnight?
operational_shifts_total = (shift_running_df[shift_running_cols] == "Operational").sum().sum()
total_production_hours = operational_shifts_total * 8
parts_per_hour = total_actual_output / total_production_hours if total_production_hours > 0 else 0
answer_9 = int(parts_per_hour)
print("Question 9:", answer_9)

# 10. The company uses MAPE (Mean Absolute Percentage Error) to measure process variability in the manufacturing process. Which shift sees the maximum process variability during the fortnight?
rated_output = 4000
shift_data = []
for i, (shift_running_col, output_col) in enumerate(zip(shift_running_cols, output_cols)):
    operational_mask = shift_running_df[shift_running_col] == "Operational"
    if operational_mask.sum() > 0:
        actual_outputs = actual_output_df.loc[operational_mask, output_col]
        mape = (abs(actual_outputs - rated_output) / rated_output * 100).mean()
        shift_data.append((f"Shift {i+1}", mape))

max_mape_shift = max(shift_data, key=lambda x: x[1])[0]
answer_10 = max_mape_shift
print("Question 10:", answer_10)